<a href="https://colab.research.google.com/github/manaskanugo97-max/Healthcare-Lead-Gen-System/blob/main/osm_scraper_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/healthcare_lead_gen_project"

DATA_DIR = os.path.join(PROJECT_DIR, "data")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print("Project folder created:")
print(PROJECT_DIR)

print("\nData folder created:")
print(DATA_DIR)

Project folder created:
/content/drive/MyDrive/healthcare_lead_gen_project

Data folder created:
/content/drive/MyDrive/healthcare_lead_gen_project/data


In [ ]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
import os
print(os.getcwd())

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
%%writefile config.py

import os

PROJECT_DIR = "/content/drive/MyDrive/healthcare_lead_gen_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

RAW_OSM_CSV = os.path.join(DATA_DIR, "healthcare_osm_raw.csv")
ONLINE_PRESENCE_CSV = os.path.join(DATA_DIR, "healthcare_online_presence.csv")
FINAL_CSV = os.path.join(DATA_DIR, "healthcare_leads_final.csv")

Overwriting config.py


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/healthcare_lead_gen_project"))

['data', 'config.py']


In [ ]:
from config import PROJECT_DIR, DATA_DIR, RAW_OSM_CSV, ONLINE_PRESENCE_CSV, FINAL_CSV

print("Project folder:", PROJECT_DIR)
print("Data folder:", DATA_DIR)
print("Step 1 output CSV:", RAW_OSM_CSV)
print("Step 2 output CSV:", ONLINE_PRESENCE_CSV)
print("Final CSV:", FINAL_CSV)

Project folder: /content/drive/MyDrive/healthcare_lead_gen_project
Data folder: /content/drive/MyDrive/healthcare_lead_gen_project/data
Step 1 output CSV: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_osm_raw.csv
Step 2 output CSV: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_online_presence.csv
Final CSV: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_leads_final.csv


In [ ]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
import os
print(os.getcwd())
print(os.listdir())

/content/drive/MyDrive/healthcare_lead_gen_project
['data', 'config.py', '__pycache__']


In [ ]:
!pip install overpy pandas -q

In [ ]:
%%writefile osm_scraper.py

import overpy
import pandas as pd
import time
from config import RAW_OSM_CSV


OVERPASS_ENDPOINTS = [
    "https://overpass.private.coffee/api/interpreter",
    "https://overpass-api.de/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter"
]


def get_value(tags, possible_keys):
    for key in possible_keys:
        if key in tags and tags[key]:
            return tags[key]
    return ""


def get_lat_lon(element):
    lat = getattr(element, "lat", None)
    lon = getattr(element, "lon", None)

    if lat is None:
        lat = getattr(element, "center_lat", None)
    if lon is None:
        lon = getattr(element, "center_lon", None)

    return lat, lon


def create_google_maps_link(lat, lon):
    if lat and lon:
        return f"https://www.google.com/maps/search/?api=1&query={lat},{lon}"
    return ""


def create_osm_link(element):
    element_type = element.__class__.__name__.lower()

    if "node" in element_type:
        return f"https://www.openstreetmap.org/node/{element.id}"
    elif "way" in element_type:
        return f"https://www.openstreetmap.org/way/{element.id}"
    elif "relation" in element_type:
        return f"https://www.openstreetmap.org/relation/{element.id}"
    return ""


def run_overpass_query(query):
    last_error = None

    for endpoint in OVERPASS_ENDPOINTS:
        try:
            print(f"Trying Overpass endpoint: {endpoint}")
            api = overpy.Overpass(url=endpoint)
            result = api.query(query)
            print("Endpoint working.")
            return result

        except Exception as e:
            print(f"Endpoint failed: {endpoint}")
            print(f"Reason: {e}")
            last_error = e
            time.sleep(3)

    raise Exception(f"All Overpass endpoints failed. Last error: {last_error}")


def scrape_healthcare_from_osm(city_name="Indore", output_file=RAW_OSM_CSV):
    query = f"""
    [out:json][timeout:90];

    area["name"="{city_name}"]["boundary"="administrative"]->.searchArea;

    (
      node["amenity"~"hospital|clinic|doctors|dentist|pharmacy"](area.searchArea);
      way["amenity"~"hospital|clinic|doctors|dentist|pharmacy"](area.searchArea);
      relation["amenity"~"hospital|clinic|doctors|dentist|pharmacy"](area.searchArea);

      node["healthcare"~"hospital|clinic|doctor|dentist|pharmacy|laboratory|physiotherapist"](area.searchArea);
      way["healthcare"~"hospital|clinic|doctor|dentist|pharmacy|laboratory|physiotherapist"](area.searchArea);
      relation["healthcare"~"hospital|clinic|doctor|dentist|pharmacy|laboratory|physiotherapist"](area.searchArea);
    );

    out center tags;
    """

    print(f"Collecting healthcare data from OpenStreetMap for: {city_name}")
    print(f"Output file will be saved at: {output_file}")

    result = run_overpass_query(query)

    all_elements = result.nodes + result.ways + result.relations

    records = []

    for element in all_elements:
        tags = element.tags

        business_name = get_value(tags, ["name"])

        if not business_name:
            continue

        category = get_value(tags, ["amenity", "healthcare"])
        lat, lon = get_lat_lon(element)

        phone = get_value(tags, ["phone", "contact:phone", "mobile", "contact:mobile"])
        email = get_value(tags, ["email", "contact:email"])
        website = get_value(tags, ["website", "contact:website", "url"])

        address_parts = [
            get_value(tags, ["addr:housenumber"]),
            get_value(tags, ["addr:street"]),
            get_value(tags, ["addr:suburb"]),
            get_value(tags, ["addr:city"]),
            get_value(tags, ["addr:postcode"])
        ]

        address = ", ".join([part for part in address_parts if part])

        if not address:
            address = city_name

        description = get_value(tags, ["description"])

        if not description:
            description = f"{category.title()} listed on OpenStreetMap"

        record = {
            "Business Name": business_name,
            "Industry Category": category,
            "Business Description": description,
            "Location": address,
            "Latitude": lat,
            "Longitude": lon,
            "Google Maps Profile Link": create_google_maps_link(lat, lon),
            "OpenStreetMap Link": create_osm_link(element),
            "Website URL": website,
            "Phone Number": phone,
            "Email Address": email,
            "Source": "OpenStreetMap"
        }

        records.append(record)

    df = pd.DataFrame(records)

    if not df.empty:
        df.drop_duplicates(
            subset=["Business Name", "Latitude", "Longitude"],
            inplace=True
        )

    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Scraping completed.")
    print(f"Total records collected: {len(df)}")
    print(f"Saved file: {output_file}")

    return df


if __name__ == "__main__":
    df = scrape_healthcare_from_osm(
        city_name="Indore",
        output_file=RAW_OSM_CSV
    )

    print(df.head())

Writing osm_scraper.py


In [ ]:
!python osm_scraper.py

Output file will be saved at: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_osm_raw.csv
Trying Overpass endpoint: https://overpass.private.coffee/api/interpreter
Endpoint failed: https://overpass.private.coffee/api/interpreter
Reason: Server load too high
Trying Overpass endpoint: https://overpass-api.de/api/interpreter
Endpoint failed: https://overpass-api.de/api/interpreter
Reason: Unknown/Unhandled status code: 406
Trying Overpass endpoint: https://maps.mail.ru/osm/tools/overpass/api/interpreter
Endpoint working.
Scraping completed.
Total records collected: 2288
Saved file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_osm_raw.csv
                                       Business Name  ...         Source
0                           Greater Kailash Hospital  ...  OpenStreetMap
1                                      Qurban Husain  ...  OpenStreetMap
2  Shri Aurobindo dental clinic and institute of ...  ...  OpenStreetMap
3                       

In [ ]:
import os
from config import RAW_OSM_CSV

print("CSV path:", RAW_OSM_CSV)
print("File exists:", os.path.exists(RAW_OSM_CSV))

CSV path: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_osm_raw.csv
File exists: True


In [ ]:
import pandas as pd
from config import RAW_OSM_CSV

df = pd.read_csv(RAW_OSM_CSV)
df.head()

,Business Name,Industry Category,Business Description,Location,Latitude,Longitude,Google Maps Profile Link,OpenStreetMap Link,Website URL,Phone Number,Email Address,Source
0,Greater Kailash Hospital,hospital,Hospital listed on OpenStreetMap,Indore,22.725082,75.890465,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/1802056893,NaN,NaN,NaN,OpenStreetMap
1,Qurban Husain,doctors,Doctors listed on OpenStreetMap,Indore,22.716741,75.863790,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/3360065915,NaN,NaN,NaN,OpenStreetMap
2,Shri Aurobindo dental clinic and institute of ...,clinic,Clinic listed on OpenStreetMap,Ujjain Road,22.791523,75.845636,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4273145396,NaN,NaN,NaN,OpenStreetMap
3,Dr. Mutha,doctors,Doctors listed on OpenStreetMap,Gita Bhavan intersection,22.718593,75.885588,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4310485189,NaN,+9198-26-150111,NaN,OpenStreetMap
4,Arshid shah,pharmacy,Pharmacy listed on OpenStreetMap,"Dhar road, 452001",22.709624,75.827079,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4522507595,NaN,8103012446,shah.arshid05@rediffmail.com,OpenStreetMap


In [ ]:
print("Total records:", len(df))

Total records: 2288
